## WorldView Stereopair Processing: SpaceNet UCSD Example — 21° convergence, 12-day pair

This notebook demonstrates a stereo processing workflow for WorldView-3 satellite imagery using the NASA Ames Stereo Pipeline (ASP) on one of three pairs re-selected from the [IARPA CORE3D SpaceNet UCSD dataset](https://spacenet.ai/core3d/).

Pair `21deg_12d`:

| catid | date | off-nadir (°) | |
|---|---|---|---|
| `1040010007A93700` | 2015-02-12 | 8.4 | left |
| `1040010007CA4D00` | 2015-02-24 | 12.9 | right |

Pair geometry (from `StereopairMetadataParser`):

- **Convergence angle**: 21.21°
- **Base-to-height ratio**: 0.37
- **BIE angle**: 84.31° (ideal 90°)
- **Asymmetry angle**: 2.73° (ideal 0°)
- **Temporal separation**: 12 days
- **Sun-elevation delta**: 3.1°
- **Intersection overlap**: 94.8%

### Why this pair?

UCSD is a mixed landscape — campus and residential buildings plus coastal valleys and hills — so we target a **~15–25° convergence angle**: enough base-to-height to constrain heights over the natural portions, but not so wide that occlusion dominates on building facades. See the [scene-selection notebook](./worldview_spacenet_ucsd_stereo_scene_selection.ipynb) for the full pair-ranking analysis.

The pipeline below (bundle_adjust → mapproject → parallel_stereo → point2dem → geodiff) is shared across all three pair notebooks and runs on the **same 3 × 3 km ROI**, so the resulting DEMs are directly comparable.

---

### Processing Overview

- **Data Retrieval** — Download WorldView-3 imagery from AWS S3
- **Reference DEM Preparation** — Obtain the Copernicus 30m DEM with ellipsoid heights (WGS84)
- **Bundle Adjustment** — Camera model refinement on full images
- **Mapprojection** — Project images onto reference DEM over the 3 × 3 km ROI
- **Stereo Processing** — Generate DEM from the stereopair
- **Visualization with `asp_plot`** — Plots, ICESat-2 comparison, PDF report


---

## Data Retrieval

The full catalogue of WV3 scenes and the pair-selection analysis is documented in the companion notebook [`worldview_spacenet_ucsd_stereo_scene_selection.ipynb`](./worldview_spacenet_ucsd_stereo_scene_selection.ipynb). This notebook processes the **`21deg_12d`** pair chosen there.


### Download Commands

Create a working directory and download the NTF images and metadata archives:

```bash
$ mkdir -p ucsd_stereo_21deg_12d/images
$ cd ucsd_stereo_21deg_12d/images

# Image 1: 2015-02-12, off-nadir (8.4°), WV3, catid 1040010007A93700
$ aws s3 --no-sign-request cp \
  s3://spacenet-dataset/Hosted-Datasets/CORE3D-Public-Data/Satellite-Images/UCSD/WV3/PAN/12FEB15WV031300015FEB12183926-P1BS-500647760030_01_P001_________AAE_0AAAAABPABS0.NTF .
$ aws s3 --no-sign-request cp \
  s3://spacenet-dataset/Hosted-Datasets/CORE3D-Public-Data/Satellite-Images/UCSD/WV3/PAN/12FEB15WV031300015FEB12183926-P1BS-500647760030_01_P001_________AAE_0AAAAABPABS0.tar .

# Image 2: 2015-02-24, off-nadir (12.9°), WV3, catid 1040010007CA4D00
$ aws s3 --no-sign-request cp \
  s3://spacenet-dataset/Hosted-Datasets/CORE3D-Public-Data/Satellite-Images/UCSD/WV3/PAN/24FEB15WV031300015FEB24183134-P1BS-500647759040_01_P001_________AAE_0AAAAABPABS0.NTF .
$ aws s3 --no-sign-request cp \
  s3://spacenet-dataset/Hosted-Datasets/CORE3D-Public-Data/Satellite-Images/UCSD/WV3/PAN/24FEB15WV031300015FEB24183134-P1BS-500647759040_01_P001_________AAE_0AAAAABPABS0.tar .

# Extract metadata from tars
$ for f in *.tar; do tar xf "$f"; done
```

After extraction, the NTF and XML files are renamed to `<CATID>_P001.{NTF,xml}` at the top level of the working directory, which keeps the ASP command invocations below short and explicit.


### Stereo Geometry Analysis

Before processing, it's useful to analyze the stereo acquisition geometry to verify the convergence angle and other geometric properties. We use `asp_plot` to visualize the geometry from the XML camera metadata:

In [ ]:
# Set the base directory for your processing
directory = "~/Desktop/asp-plot-examples/ucsd_stereo_21deg_12d/"

# Create geometry plotter and auto-detect UTM projection and scene extent from XML metadata
from asp_plot.stereo_geometry import StereoGeometryPlotter

sgp = StereoGeometryPlotter(directory)
utm_code = sgp.get_pair_utm_epsg()
map_crs = f"EPSG:{utm_code}"
scene_bounds = sgp.get_scene_bounds()

print(f"Auto-detected UTM projection: {map_crs}")
print(f"Scene extent (lon/lat): {scene_bounds[0]:.6f} {scene_bounds[1]:.6f} {scene_bounds[2]:.6f} {scene_bounds[3]:.6f}")

# Generate the geometry plot
sgp.dg_geom_plot()


### Satellite Position and Orientation Data

The XML camera metadata also contains ephemeris (position/velocity) and attitude (orientation quaternion) data reported by the satellite during acquisition. Visualizing this data can help assess the quality of the raw metadata before ASP processing.

In [ ]:
sgp.satellite_position_orientation_plot()

In the above plot:

- Top row (Position Covariance): A map showing the satellite's path over the ground during image capture, colored by how uncertain the satellite's reported position is (in meters). Lower values mean the satellite knows where it is more precisely.

- Middle row (Roll / Pitch / Yaw): Shows how the satellite's pointing deviates from its expected nadir-pointing orientation over time. Roll is rotation around the flight direction, pitch is rotation around the cross-track axis, and yaw is rotation around the nadir axis. Values near zero mean the satellite is pointing straight down. These are computed by comparing the raw attitude quaternions against a reference orientation estimated from the satellite's orbital position and velocity.

- Bottom row (Attitude Covariance Trace): Shows how uncertain the satellite's orientation knowledge is over time. Lower values mean the satellite is more confident about which direction it's pointing. Spikes or jumps indicate moments of less reliable pointing knowledge, which could affect image quality in those portions of the scene.

---

## CCD Artifact Correction — Not Needed for WorldView-3

The [`wv_correct`](https://stereopipeline.readthedocs.io/en/latest/tools/wv_correct.html) tool is used to correct subpixel CCD boundary misalignment artifacts in WorldView-1 and WorldView-2 imagery. These artifacts manifest as discontinuities in DEMs at CCD boundaries.

**WorldView-3 does not suffer from the same CCD boundary artifacts**, so `wv_correct` is **not needed** for this dataset and we skip this step entirely.

For comparison, see the [Atlanta example](https://asp-plot.readthedocs.io/en/latest/examples/notebooks/worldview_spacenet_atlanta_stereo.html) where `wv_correct` is applied to WorldView-2 imagery.

---

## Processing Configuration

```bash
$ cd ucsd_stereo_21deg_12d

# Target spatial reference system (UTM Zone 11N for San Diego)
$ t_srs="EPSG:32611"

# Processing ROI: 3 × 3 km over UCSD campus + Mount Soledad
$ t_projwin="476000 3635600 479000 3638600"

# Output resolution for mapproject — use the more nadir image's meanProductGSD
$ tr=0.315

# Reference DEM filename (WGS84 ellipsoid heights, projected to UTM)
$ reference_dem_fn="ref/cop30_ucsd_wgs84_utm.tif"
```


---

## Reference DEM Preparation

A reference DEM is required for:
- Mapprojecting the input images
- Validating the output DEM

We use the Copernicus 30m GLO-30 DEM, which provides global coverage with good accuracy.

### Download Copernicus DEM with Ellipsoid Heights

We download with [fetch_dem](https://github.com/uw-cryo/fetch_dem). This requires a free [OpenTopography API key](https://portal.opentopography.org/requestService?service=api).

**Important:** We use `-demtype COP30_E` to download the DEM with **WGS84 ellipsoid heights** directly. ASP requires heights relative to the WGS84 ellipsoid.

The `-extent` argument uses the `scene_bounds` from the auto-detected union of both image footprints above. Ensure that the extent has full coverage over the area you are attempting to process.

```bash
$ cd ucsd_stereo_21deg_12d

# Download Copernicus DEM with ellipsoid heights (COP30_E)
$ python /path/to/your/fetch_dem/download_global_DEM.py \
  -demtype COP30_E \
  -extent "$scene_bounds" \
  -out_fn "$reference_dem_fn" \
  -out_proj "$t_srs" \
  -apikey [YOUR_OPEN_TOPOGRAPHY_API_KEY]
```

This creates the reference DEM at `ref/cop30_ucsd_wgs84_utm.tif` with WGS84 ellipsoid heights in UTM Zone 11N projection.


---

## Bundle Adjustment

Bundle adjustment refines camera models by minimizing reprojection errors of matched feature points. Run on the **full** images (no ROI cropping) so tie points are spread across the whole scene:

```bash
$ cd ucsd_stereo_21deg_12d

$ bundle_adjust \
  --threads 8 \
  --ip-per-image 10000 \
  --tri-weight 0.1 \
  --tri-robust-threshold 0.1 \
  --camera-weight 0 \
  1040010007A93700_P001.NTF 1040010007CA4D00_P001.NTF \
  1040010007A93700_P001.xml 1040010007CA4D00_P001.xml \
  -o ba/run
```


### Bundle Adjustment Results

Visualize the bundle adjustment results to verify the optimization reduced reprojection errors:

In [ ]:
from asp_plot.bundle_adjust import ReadBundleAdjustFiles, PlotBundleAdjustFiles
import contextily as ctx

# Define subdirectories
bundle_adjust_directory = "ba/"
stereo_directory = "stereo/"

# Map configuration (map_crs was auto-detected in the geometry cell above)
ctx_kwargs = {
    "crs": map_crs,
    "source": ctx.providers.Esri.WorldImagery,
    "attribution_size": 0,
    "alpha": 0.5,
}

# Read bundle adjustment residuals
ba_files = ReadBundleAdjustFiles(directory, bundle_adjust_directory)
resid_initial_gdf, resid_final_gdf = ba_files.get_initial_final_residuals_gdfs(residuals_in_meters=True)

# Plot residuals before and after optimization
plotter = PlotBundleAdjustFiles(
    [resid_initial_gdf, resid_final_gdf],
    lognorm=True,
    title="Bundle Adjust Initial and Final Residuals"
)
plotter.plot_n_gdfs(
    column_name="mean_residual",
    cbar_label="Mean residual (px)",
    map_crs=map_crs,
    **ctx_kwargs
)

---

## Define Region of Interest

The full UCSD images are very large (~43,000 x 43,000 pixels each). To make processing tractable, we define a **3 × 3 km ROI** covering the UCSD campus plus the steep terrain of Mount Soledad and Torrey Pines to the southwest toward La Jolla. The same ROI is used across all three pair notebooks (14°, 18°, 21°) so their DEMs can be compared directly.

```bash
# ROI in UTM Zone 11N (EPSG:32611): xmin ymin xmax ymax
$ t_projwin="476000 3635600 479000 3638600"
```

The [scene-selection notebook](./worldview_spacenet_ucsd_stereo_scene_selection.ipynb) verified that this ROI is fully contained in this pair's image intersection.


In [ ]:
# Get the intersection extent in UTM coordinates (from the geometry cell above)
intersection_bounds = sgp.get_intersection_bounds(epsg=utm_code)
t_projwin = f"{intersection_bounds[0]:.0f} {intersection_bounds[1]:.0f} {intersection_bounds[2]:.0f} {intersection_bounds[3]:.0f}"
print(f"Intersection projwin: {t_projwin}")

As noted above, we use the ROI `t_projwin="476000 3635600 479000 3638600"` — a 3 × 3 km window over UCSD campus + Mount Soledad.


---

## Mapprojection

Mapproject the two bundle-adjusted images onto the reference DEM over the shared 3 × 3 km ROI:

```bash
$ cd ucsd_stereo_21deg_12d

$ mapproject \
  -t rpc --processes 4 --threads 2 \
  --tr $tr --t_projwin $t_projwin --t_srs "$t_srs" \
  --bundle-adjust-prefix ba/run \
  "$reference_dem_fn" \
  1040010007A93700_P001.NTF 1040010007A93700_P001.xml \
  1040010007A93700_P001_map.tif

$ mapproject \
  -t rpc --processes 4 --threads 2 \
  --tr $tr --t_projwin $t_projwin --t_srs "$t_srs" \
  --bundle-adjust-prefix ba/run \
  "$reference_dem_fn" \
  1040010007CA4D00_P001.NTF 1040010007CA4D00_P001.xml \
  1040010007CA4D00_P001_map.tif
```


---

## Stereo Processing

```bash
$ cd ucsd_stereo_21deg_12d

$ parallel_stereo \
  --stereo-algorithm asp_mgm --subpixel-mode 9 \
  --processes 2 --threads 4 --alignment-method none \
  --bundle-adjust-prefix ba/run \
  1040010007A93700_P001_map.tif 1040010007CA4D00_P001_map.tif \
  1040010007A93700_P001.xml 1040010007CA4D00_P001.xml \
  stereo/run "$reference_dem_fn"

# Generate ~1.2 m DEM (~4× the native input GSD)
$ point2dem --tr 1.2 --t_srs "$t_srs" --errorimage stereo/run-PC.tif

# Difference map vs. reference DEM
$ geodiff stereo/run-DEM.tif "$reference_dem_fn" -o stereo/run_vs_ref
```


### Stereo Results Visualization

Examine the stereo processing outputs to assess DEM quality:

In [ ]:
from asp_plot.stereo import StereoPlotter
from asp_plot.scenes import ScenePlotter

# Plot input scenes (mapprojected images)
scene_plotter = ScenePlotter(directory, stereo_directory, title="Mapprojected Input Scenes")
scene_plotter.plot_scenes()

# Plot DEM results
stereo_plotter = StereoPlotter(directory, stereo_directory)
stereo_plotter.title = "Stereo DEM Results"
stereo_plotter.plot_dem_results()

# Plot detailed hillshade with zoom subsets
stereo_plotter.title = "Hillshade with Details"
stereo_plotter.plot_detailed_hillshade(subset_km=0.5)

---

## Comprehensive Report and ICESat-2 Altimetry Validation

In addition to the inline visualizations above, we can validate the ASP DEM against ICESat-2 ATL06-SR altimetry data. This provides an independent accuracy assessment using satellite laser altimetry.

The sections below demonstrate ICESat-2 comparison and generate a comprehensive PDF report.

### Setup

After you have `asp_plot` installed and ready to use, you can set up the directory and file names:

In [ ]:
# Set the base directory for your processing
directory = "~/Desktop/asp-plot-examples/ucsd_stereo_21deg_12d/"

# Define subdirectories for bundle adjustment and stereo results
bundle_adjust_directory = "ba/"
stereo_directory = "stereo/"


### Full Report Generation

Generate a comprehensive PDF report with a single CLI command:

#### **Note:** The generated report is saved to `reports/WorldView_UCSD_21deg_12d-asp-plot-report.pdf`.


In [ ]:
!asp_plot \
  --directory $directory \
  --bundle_adjust_directory $bundle_adjust_directory \
  --stereo_directory $stereo_directory \
  --map_crs $map_crs \
  --plot_altimetry True \
  --atl06sr_time_range "all" \
  --plot_geometry True \
  --report_filename ../../reports/WorldView_UCSD_21deg_12d-asp-plot-report.pdf


### ICESat-2 Altimetry Validation

Compare the ASP DEM with ICESat-2 ATL06-SR altimetry data:

In [ ]:
from asp_plot.altimetry import Altimetry

In [ ]:
icesat = Altimetry(
  directory=directory,
  dem_fn=stereo_plotter.dem_fn
)

In [ ]:
# Request ATL06-SR data (single "all" processing level)
icesat.request_atl06sr_multi_processing(
    processing_levels=["all"],
    save_to_parquet=True,
)

In [ ]:
# Filter out water bodies
icesat.filter_esa_worldcover(filter_out="water")

No temporal filtering needed for the simplified report workflow.
Advanced temporal filtering is still available via:

```py
icesat.predefined_temporal_filter_atl06sr(date=...)
icesat.generic_temporal_filter_atl06sr(select_years=..., select_months=..., ...)
```

In [ ]:
# Map view of ICESat-2 vs DEM differences
icesat.mapview_plot_atl06sr_to_dem(
    key="all",
    map_crs=map_crs,
    **ctx_kwargs,
)

In [ ]:
# Histogram with per-landcover-class statistics
icesat.histogram_by_landcover(key="all")

In [ ]:
# Profile along the best ICESat-2 track
icesat.plot_atl06sr_dem_profile(key="all")
icesat.plot_best_worst_segments(key="all")

### DEM-to-Altimetry Alignment

Run `pc_align` to compute the translation required to align the DEM with ICESat-2 data:

**Note on multiple pc_align runs:** The `alignment_report` method runs `pc_align` once for each temporal filter (e.g., "ground", "ground_seasonal", etc.). This allows comparison of alignment results across different subsets of ICESat-2 data. Each row in the resulting DataFrame represents a separate alignment run.

The output columns include:
- `north_shift`, `east_shift`, `down_shift`: Translation vector in North-East-Down coordinates (meters)
- `p16_beg/end`, `p50_beg/end`, `p84_beg/end`: Error percentiles before and after alignment

In [ ]:
# Run alignment report to assess DEM quality
icesat.alignment_report(
    processing_level="all",
    agreement_threshold=0.25,
    write_out_aligned_dem=True,
)

**Note:** if not enough points are available to do the alignment we can still attempt it by lowering the minimum points. Use results with caution when aligning to fewer points:

```py
# Run alignment report to assess DEM quality with a minimum points threshold of 50 (instead of the default 500)
icesat.alignment_report(
    processing_level="all",
    minimum_points=50,
    agreement_threshold=0.25,
    write_out_aligned_dem=True,
)
```

In [ ]:
icesat.alignment_report_df

In [ ]:
# Histogram of aligned DEM vs ICESat-2 differences
icesat.atl06sr_to_dem_dh()

icesat.histogram(
    key="all",
    plot_aligned=True,
)

---

## Next Steps

This notebook processed one of three UCSD stereo pairs chosen via the [scene-selection notebook](./worldview_spacenet_ucsd_stereo_scene_selection.ipynb). The sibling notebooks process the other two pairs on the same ROI so their DEMs and altimetry-agreement statistics can be compared directly:

- [`worldview_spacenet_ucsd_stereo_14deg_6d.ipynb`](./worldview_spacenet_ucsd_stereo_14deg_6d.ipynb) — 14.14° convergence, 6-day separation
- [`worldview_spacenet_ucsd_stereo_18deg_13d.ipynb`](./worldview_spacenet_ucsd_stereo_18deg_13d.ipynb) — 17.52° convergence, 13-day separation
- [`worldview_spacenet_ucsd_stereo_21deg_12d.ipynb`](./worldview_spacenet_ucsd_stereo_21deg_12d.ipynb) — 21.21° convergence, 12-day separation
